In [9]:
"""Build shared 80/20 spatial and random plot-level splits.

The spatial test set is selected from all contiguous 20% northing bands that
retain presences of the four data-rich species in both train and test. Among
those valid bands, the band with the largest median test-to-nearest-train
distance is selected. The random test set uses the same number of plots.

All eight rows belonging to a plot remain together, preventing plot leakage.
"""

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd


PLOT_COL = "plot"
SPECIES_COL = "Species"
TARGET_COL = "pres.abs"
EASTING_COL = "easting"
NORTHING_COL = "northing"

TEST_FRACTION = 0.20
RANDOM_STATE = 42

# These strings must match the values in train.csv exactly.
CORE_SPECIES = [
    "Calyptotis scutirostrum",
    "Coeranoscincus reticulatus",
    "Ophioscincus truncatus",
    "Cacophis kreftii",
]


def build_plot_table(df):
    """Convert long-format species rows to one row per plot."""
    required = {
        PLOT_COL,
        SPECIES_COL,
        TARGET_COL,
        EASTING_COL,
        NORTHING_COL,
    }
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    duplicated = df.duplicated([PLOT_COL, SPECIES_COL])
    if duplicated.any():
        raise ValueError(
            f"Found {int(duplicated.sum())} duplicated plot-species rows."
        )

    coordinate_counts = df.groupby(PLOT_COL)[
        [EASTING_COL, NORTHING_COL]
    ].nunique()
    if (coordinate_counts > 1).any().any():
        raise ValueError("Some plots have inconsistent coordinates.")

    coordinates = (
        df[[PLOT_COL, EASTING_COL, NORTHING_COL]]
        .drop_duplicates(PLOT_COL)
        .copy()
    )

    presences = (
        df.pivot_table(
            index=PLOT_COL,
            columns=SPECIES_COL,
            values=TARGET_COL,
            aggfunc="max",
            fill_value=0,
        )
        .reset_index()
    )

    plot_table = coordinates.merge(
        presences,
        on=PLOT_COL,
        how="inner",
        validate="one_to_one",
    )

    missing_species = [
        species for species in CORE_SPECIES
        if species not in plot_table.columns
    ]
    if missing_species:
        raise ValueError(
            f"Core species not found: {missing_species}. "
            f"Available species: {sorted(df[SPECIES_COL].unique())}"
        )

    return plot_table


def presence_summary(plot_table, train_ids, test_ids):
    """Count core-species presences in the train and test plots."""
    train = plot_table[plot_table[PLOT_COL].isin(train_ids)]
    test = plot_table[plot_table[PLOT_COL].isin(test_ids)]

    return pd.DataFrame(
        [
            {
                "Species": species,
                "train_presence": int(train[species].sum()),
                "test_presence": int(test[species].sum()),
            }
            for species in CORE_SPECIES
        ]
    )


def is_valid_presence_split(summary):
    """Return True when every core species has presences on both sides."""
    return bool(
        (
            (summary["train_presence"] > 0)
            & (summary["test_presence"] > 0)
        ).all()
    )


def nearest_distance_table(plot_table, train_ids, test_ids):
    """Find each test plot's nearest training plot and distance in km."""
    train = (
        plot_table[plot_table[PLOT_COL].isin(train_ids)]
        [[PLOT_COL, EASTING_COL, NORTHING_COL]]
        .reset_index(drop=True)
    )
    test = (
        plot_table[plot_table[PLOT_COL].isin(test_ids)]
        [[PLOT_COL, EASTING_COL, NORTHING_COL]]
        .reset_index(drop=True)
    )

    train_xy = train[[EASTING_COL, NORTHING_COL]].to_numpy(dtype=float)
    records = []

    for _, row in test.iterrows():
        test_xy = row[[EASTING_COL, NORTHING_COL]].to_numpy(dtype=float)
        distances_m = np.sqrt(np.sum((train_xy - test_xy) ** 2, axis=1))
        nearest_position = int(np.argmin(distances_m))

        records.append(
            {
                "test_plot": row[PLOT_COL],
                "nearest_train_plot": train.iloc[nearest_position][PLOT_COL],
                "distance_km": float(distances_m[nearest_position] / 1000),
            }
        )

    return pd.DataFrame(records)


def select_spatial_band(plot_table, test_fraction):
    """Select the valid contiguous northing band with maximum separation."""
    ordered = plot_table.sort_values(
        [NORTHING_COL, EASTING_COL, PLOT_COL]
    ).reset_index(drop=True)

    n_total = len(ordered)
    n_test = int(round(n_total * test_fraction))
    if n_test <= 0 or n_test >= n_total:
        raise ValueError("test_fraction must leave plots on both sides.")

    candidates = []

    for start in range(n_total - n_test + 1):
        test_slice = ordered.iloc[start : start + n_test]
        test_ids = test_slice[PLOT_COL].tolist()
        test_id_set = set(test_ids)
        train_ids = [
            plot_id for plot_id in ordered[PLOT_COL].tolist()
            if plot_id not in test_id_set
        ]

        summary = presence_summary(plot_table, train_ids, test_ids)
        if not is_valid_presence_split(summary):
            continue

        distances = nearest_distance_table(plot_table, train_ids, test_ids)
        median_km = float(distances["distance_km"].median())

        candidates.append(
            {
                "start": start,
                "train_ids": train_ids,
                "test_ids": test_ids,
                "summary": summary,
                "distances": distances,
                "median_km": median_km,
                "northing_min": float(test_slice[NORTHING_COL].min()),
                "northing_max": float(test_slice[NORTHING_COL].max()),
            }
        )

    if not candidates:
        raise ValueError(
            "No valid contiguous northing band retained all core-species "
            "presences in both train and test."
        )

    # This rule is fixed before model fitting and never uses model scores.
    return max(candidates, key=lambda candidate: candidate["median_km"])


def make_random_split(plot_table, n_test, random_state):
    """Randomly split complete plots using the spatial test-set size."""
    rng = np.random.default_rng(random_state)
    plot_ids = plot_table[PLOT_COL].to_numpy().copy()
    rng.shuffle(plot_ids)

    test_ids = plot_ids[:n_test].tolist()
    train_ids = plot_ids[n_test:].tolist()
    summary = presence_summary(plot_table, train_ids, test_ids)

    if not is_valid_presence_split(summary):
        raise ValueError(
            f"Random seed {random_state} did not retain all core-species "
            f"presences on both sides.\n{summary.to_string(index=False)}"
        )

    distances = nearest_distance_table(plot_table, train_ids, test_ids)
    return {
        "train_ids": train_ids,
        "test_ids": test_ids,
        "summary": summary,
        "distances": distances,
        "median_km": float(distances["distance_km"].median()),
    }


def apply_split(df, train_ids, test_ids):
    """Apply plot IDs to every species row in the long-format data."""
    train = df[df[PLOT_COL].isin(train_ids)].copy()
    test = df[df[PLOT_COL].isin(test_ids)].copy()
    return train, test


def validate_split(original_df, train_df, test_df, expected_test_plots, name):
    """Fail if plots overlap, plots/rows are lost, or sizes are incorrect."""
    original_ids = set(original_df[PLOT_COL].unique())
    train_ids = set(train_df[PLOT_COL].unique())
    test_ids = set(test_df[PLOT_COL].unique())

    overlap = train_ids & test_ids
    if overlap:
        raise ValueError(f"{name}: {len(overlap)} plots overlap.")
    if train_ids | test_ids != original_ids:
        raise ValueError(f"{name}: plots were lost or added.")
    if len(test_ids) != expected_test_plots:
        raise ValueError(
            f"{name}: expected {expected_test_plots} test plots, "
            f"found {len(test_ids)}."
        )
    if len(train_df) + len(test_df) != len(original_df):
        raise ValueError(f"{name}: rows were lost or duplicated.")


def json_value(value):
    """Convert NumPy scalar values to JSON-compatible Python values."""
    return value.item() if isinstance(value, np.generic) else value


def clean_known_outputs(output_dir):
    """Delete only files this script owns; preserve unrelated files."""
    filenames = [
        "spatial_train.csv",
        "spatial_test.csv",
        "random_train.csv",
        "random_test.csv",
        "data_splits.json",
        "spatial_presence_summary.csv",
        "random_presence_summary.csv",
        "spatial_nearest_distances.csv",
        "random_nearest_distances.csv",
    ]
    for filename in filenames:
        path = output_dir / filename
        if path.exists():
            path.unlink()


def main():
    parser = argparse.ArgumentParser(
        description="Build shared 80/20 spatial and random plot-level splits."
    )
    parser.add_argument("--data", default="train.csv")
    parser.add_argument("--output-dir", default="data_splits_80_20")
    parser.add_argument("--test-fraction", type=float, default=TEST_FRACTION)
    parser.add_argument("--random-state", type=int, default=RANDOM_STATE)
    args, _ = parser.parse_known_args()  # Safe in Jupyter notebooks.

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    clean_known_outputs(output_dir)

    print(f"Loading {args.data} ...")
    df = pd.read_csv(args.data)
    plot_table = build_plot_table(df)

    spatial = select_spatial_band(plot_table, args.test_fraction)
    random = make_random_split(
        plot_table,
        n_test=len(spatial["test_ids"]),
        random_state=args.random_state,
    )

    spatial_train, spatial_test = apply_split(
        df, spatial["train_ids"], spatial["test_ids"]
    )
    random_train, random_test = apply_split(
        df, random["train_ids"], random["test_ids"]
    )

    expected_test_plots = len(spatial["test_ids"])
    validate_split(
        df, spatial_train, spatial_test, expected_test_plots, "Spatial split"
    )
    validate_split(
        df, random_train, random_test, expected_test_plots, "Random split"
    )

    spatial_train.to_csv(output_dir / "spatial_train.csv", index=False)
    spatial_test.to_csv(output_dir / "spatial_test.csv", index=False)
    random_train.to_csv(output_dir / "random_train.csv", index=False)
    random_test.to_csv(output_dir / "random_test.csv", index=False)
    spatial["summary"].to_csv(
        output_dir / "spatial_presence_summary.csv", index=False
    )
    random["summary"].to_csv(
        output_dir / "random_presence_summary.csv", index=False
    )
    spatial["distances"].to_csv(
        output_dir / "spatial_nearest_distances.csv", index=False
    )
    random["distances"].to_csv(
        output_dir / "random_nearest_distances.csv", index=False
    )

    payload = {
        "metadata": {
            "test_fraction": args.test_fraction,
            "n_total_plots": int(plot_table[PLOT_COL].nunique()),
            "n_train_plots": int(len(spatial["train_ids"])),
            "n_test_plots": int(expected_test_plots),
            "random_state": args.random_state,
            "spatial_rule": (
                "maximum median nearest-training distance among valid "
                "contiguous northing bands"
            ),
            "core_species": CORE_SPECIES,
        },
        "spatial": {
            "train_plots": [json_value(x) for x in spatial["train_ids"]],
            "test_plots": [json_value(x) for x in spatial["test_ids"]],
            "band_start_index": int(spatial["start"]),
            "test_northing_min": spatial["northing_min"],
            "test_northing_max": spatial["northing_max"],
            "median_nearest_distance_km": spatial["median_km"],
        },
        "random": {
            "train_plots": [json_value(x) for x in random["train_ids"]],
            "test_plots": [json_value(x) for x in random["test_ids"]],
            "median_nearest_distance_km": random["median_km"],
        },
    }

    with (output_dir / "data_splits.json").open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)

    print(f"Rows in long-format data: {len(df)}")
    print(f"Unique plots: {plot_table[PLOT_COL].nunique()}")
    print(f"Train plots: {len(spatial['train_ids'])}")
    print(f"Test plots: {expected_test_plots}")
    print("\nSpatial presence summary:")
    print(spatial["summary"].to_string(index=False))
    print("\nRandom presence summary:")
    print(random["summary"].to_string(index=False))
    print(
        f"\nSpatial median nearest distance: {spatial['median_km']:.3f} km"
    )
    print(f"Random median nearest distance: {random['median_km']:.3f} km")
    print(f"\nAll checks passed. Files saved to: {output_dir.resolve()}")


if __name__ == "__main__":
    main()

Loading train.csv ...
Rows in long-format data: 5128
Unique plots: 641
Train plots: 513
Test plots: 128

Spatial presence summary:
                   Species  train_presence  test_presence
   Calyptotis scutirostrum              37             13
Coeranoscincus reticulatus              64             40
    Ophioscincus truncatus             101             13
          Cacophis kreftii              19              1

Random presence summary:
                   Species  train_presence  test_presence
   Calyptotis scutirostrum              34             16
Coeranoscincus reticulatus              84             20
    Ophioscincus truncatus              88             26
          Cacophis kreftii              16              4

Spatial median nearest distance: 18.184 km
Random median nearest distance: 0.865 km

All checks passed. Files saved to: C:\Users\13533\data_splits_80_20
